# E1 · W6 · Kaggle run 14 — video → YOLO11n → ByteTrack-e1, detection thô, GT và ghép theo frame (GPU)
**Mục đích (issue #74, ADR-0007 bản cập nhật, ADR-0009):** chạy detector đã chọn sau 3 seed (YOLO11n `s20261030`) với tracker đã giữ (ByteTrack-e1) trên **toàn bộ frame** của 38 video khả kiến tập **val** trong `e1-yolo-v1`. Test vẫn khóa.

Ngoài track log và độ đo liên kết như run 12, job lưu thêm:
- detection thô mỗi frame, dùng cho luật cảnh báo detector-only R0;
- GT theo frame và meta video (fps, kích thước), dùng để dựng sự kiện GT;
- file ghép GT–track theo frame, dùng để định vị từng ID switch (giới hạn An nêu ở #61).

Đánh giá cảnh báo chạy **ngoài Kaggle** trên các file CSV này, sau khi An duyệt ADR-0009 (#75).

**Trước khi chạy:**
1. Dataset `e1-w3-tools` → **New Version**, thay `w5_track_videos.py` bằng bản ở commit `COMMIT`. Giữ `w5_track_metrics.py`, `w3_export_yolo.py`, `annotation_audit.csv`, `tracker_bytetrack_e1.yaml`.
2. *Add Input*:
   - `e1-svanstrom-raw`;
   - `e1-w3-tools` (bản mới);
   - output notebook `e1-w3-run03b` (lấy `yolo_manifest.csv.gz`);
   - output notebook `e1-w5-run13` (lấy `best.pt` của `yolo11n_640_s20261030`).
3. *Settings*: **GPU T4 x2**, Internet **On**. Điền `COMMIT`.

**Sau khi chạy:** tải `w6_run14_evidence.zip` về thư mục kết quả run 14. `w6_run14_demo.zip` (mp4) đưa lên Drive, **không** đưa vào Git.


In [ ]:
COMMIT = ""                      # <-- SHA day du 40 ky tu
TRACKERS = ["tracker_bytetrack_e1.yaml"]       # tracker da giu theo ADR-0008 (run 12b)
WEIGHTS_SHA = "a3de1e0195e4380aac497612296ed57f67e41ceb5ae3314e82a5ad66b125ef8a"  # best.pt yolo11n_640_s20261030 (run 13)
ULTRALYTICS_VERSION = "8.4.183"  # giu dung ban cua run 12/13


In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
!pip -q install mcos-decoder "lap>=0.5.12" {"ultralytics==" + ULTRALYTICS_VERSION}
import mcos_decoder, lap, torch, ultralytics
print("ultralytics", ultralytics.__version__, "| lap", lap.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU T4 x2"

In [ ]:
# Tim thu muc du lieu tho; neu dataset tho la file .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        if "e1-yolo" in z:
            continue
        print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
SCRIPT = find("w5_track_videos.py")
for n in ["w5_track_metrics.py", "w3_export_yolo.py"]:
    assert os.path.dirname(find(n)) == os.path.dirname(SCRIPT), f"{n} phai nam cung thu muc voi w5_track_videos.py"
MANIFEST = find("yolo_manifest.csv.gz"); AUDIT = find("annotation_audit.csv")
CFGS = [find(t) for t in TRACKERS]
W = [p for p in sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True)) if sha(p) == WEIGHTS_SHA]
assert W, "Khong thay best.pt co SHA-256 cua yolo11n_640_s20261030 — kiem tra input e1-w5-run13"
WEIGHTS = W[0]
for p in (SCRIPT, MANIFEST, AUDIT, WEIGHTS, *CFGS):
    print(f"sha256={sha(p)[:16]}  {p}")

In [ ]:
!{sys.executable} {SCRIPT} --data-root {DATA_ROOT} --manifest {MANIFEST} --audit {AUDIT} --weights {WEIGHTS} --weights-sha {WEIGHTS_SHA} --trackers {" ".join(CFGS)} --out /kaggle/working --commit {COMMIT} --tag w6_run14 --save-detections --save-gt --save-matches


## Tự kiểm tra (phải in `PASS`)

In [ ]:
import pandas as pd
out = "/kaggle/working/w6_run14"
st = json.load(open(f"{out}/stage_status.json")); print({k: v["status"] for k, v in st.items()})
s = json.load(open(f"{out}/track_summary.json"))["summary"]
print(pd.DataFrame({k: v["all"] for k, v in s.items()}).T[["videos", "gt_frames", "track_recall_pooled", "id_switches",
      "idsw_per_1000_gt_frames", "videos_single_id", "median_dominant_id_coverage", "false_tracks"]].to_string())
print(pd.read_csv(f"{out}/timing.csv").groupby("tracker")["fps"].median())
ok = all(v["status"] == "OK" for v in st.values()) and list(st) == ["env", "inputs", "track", "metrics", "detect", "dump", "demo", "bundle"]
print("PASS" if ok else "FAIL - gui stage_status.json cho Claude")
print("detections:", st["detect"]["detections"], "| match files:", st["dump"]["match_files"])
for z in ("w6_run14_evidence.zip", "w6_run14_demo.zip"):
    p = f"/kaggle/working/{z}"; print(z, os.path.getsize(p) // 1024, "KB  sha256", sha(p))